In [1]:
# Logging
import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings(action='ignore')

# Typing
from typing import Any, Generator, Iterable, Sequence, Optional, Union

# Stdlib
from ast import literal_eval

# I/O
import csv
import json
from pathlib import Path

# Numeric
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem

INFO:numexpr.utils:Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:rdkit:Enabling RDKit 2023.09.6 jupyter extensions


## Formatting training data and breaking into jobs

In [15]:
MONO_DATA_DIR = Path('monomer_data_raw')
N_TO_SAMPLE : Optional[int] = 10
# N_TO_SAMPLE : Optional[int] = None

mono_data_file_name = '20231114_polyid_data_density_DP2-6 - 1,2 monomers.csv'
mono_data_file_name = 'nipu_urethanes.xlsx'
mono_data_path = MONO_DATA_DIR / mono_data_file_name
assert(mono_data_path.exists())

READERS_BY_EXT = {
    '.xlsx' : pd.read_excel,
    '.csv'  : pd.read_csv,
}
df_reader_fn = READERS_BY_EXT[mono_data_path.suffix] # don't use get() here; WANT a KeyError if invalid
mono_df = df_reader_fn(mono_data_path)

if N_TO_SAMPLE is not None:
    # mono_df = mono_df.head(min(N_TO_SAMPLE, len(mono_df)))
    mono_df = mono_df.sample(N_TO_SAMPLE)

In [16]:
mono_df

,Chemistry,Monomers
102,urethane,"('O=C=NCCCCCCN=C=O', 'OCCCCCCCCCCO')"
66,urethane,"('OCC1CCC(CO)CC1', 'O=C=NCC1CCCC(CN=C=O)C1')"
43,NIPU,"('NCc1cccc(CN)c1', 'CCC1OC(=O)OC1CC1OC(=O)OC1C..."
60,urethane,"('O=C=NCCCCN=C=O', 'OCCN(CCO)c1ccccc1')"
25,NIPU,('O=C1OCC(COCC(O)C(OCC2COC(=O)O2)C(OCC2COC(=O)...
24,NIPU,"('NCCCCCCN', 'CCCCCCCCC1OC(=O)OC1CCCCCCCC(=O)O..."
87,urethane,"('O=C=NCC1CCCC(CN=C=O)C1', 'OCC1CC2C3CC(CO)C(C..."
53,urethane,"('O=C=NCCCCCCCCCCN=C=O', 'CCCCCCC(O)CC=CCCCCCC..."
49,NIPU,"('NCCOCCOCCN', 'O=C1OCC(COc2cccc(OCC3COC(=O)O3..."
1,NIPU,('NCCCCCCNC(=O)CC(O)(CC(=O)NCCCCCCN)C(=O)NCCCC...


#### Validation functions

## Format SMILES string into a standardized format (with unit tests)

In [17]:
from polymerist.smileslib.primitives import is_valid_SMILES, is_valid_SMARTS
from polymerist.genutils.textual.encoding import hash_as_alphanumeric


def parse_monomer_smiles(smiles : Union[str, Sequence[str]]) -> Optional[str]:
    '''Enforces formatting of SMILES monomer inputs as a single dot-bond 
    joined string of the individual SMILES in canonical form'''
    if isinstance(smiles, str): # convert Sequences saved as strings to literal Sequences (i.e.  "('A', 'B')" -> ('A', 'B'))
        try:
            smiles = literal_eval(smiles)
        except (SyntaxError,):
            pass
            print('SMILES stayed as ', smiles)
    
    # print('#', smiles)
    if isinstance(smiles, Sequence) and not isinstance(smiles, str): # strings are technically Sequences, but we don't want to reformat them here
        smiles = '.'.join(smiles)

    # print('##', smiles)
    if not (isinstance(smiles, str) and is_valid_SMILES(smiles)):
        # raise TypeError
        return None
    
    return Chem.CanonSmiles(smiles)

mono_smiles_accepted = [ # all of the following format variations will be accepted
    'O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1',
    ('O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1'),
    "('O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1')",
    ['O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1'],
    "['O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1']",
]
for smi in mono_smiles_accepted:
    assert(parse_monomer_smiles(smi) is not None)

mono_smiles_invalid = [ # all of the following format variations will be accepted
    'O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1',
    ('O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1'),
    "('O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1')",
    ['O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1'],
    "['O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1']",
]
for smi in mono_smiles_invalid:
    assert(parse_monomer_smiles(smi) is None)

SMILES stayed as  O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1
SMILES stayed as  O=C(Cl)Cl.Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1
SMILES stayed as  O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1
SMILES stayed as  O=C(Cl)Cl, Oc1ccc(C(c2ccc(O)cc2)(C(F)(F)F)C(F)(F)F)cc1


[18:45:05] SMILES Parse Error: syntax error while parsing: O=C(Cl)Cl,
[18:45:05] SMILES Parse Error: Failed parsing SMILES 'O=C(Cl)Cl,' for input: 'O=C(Cl)Cl,'
[18:45:05] SMILES Parse Error: syntax error while parsing: O=C(Cl)Cl,
[18:45:05] SMILES Parse Error: Failed parsing SMILES 'O=C(Cl)Cl,' for input: 'O=C(Cl)Cl,'
[18:45:05] SMILES Parse Error: syntax error while parsing: O=C(Cl)Cl,
[18:45:05] SMILES Parse Error: Failed parsing SMILES 'O=C(Cl)Cl,' for input: 'O=C(Cl)Cl,'
[18:45:05] SMILES Parse Error: syntax error while parsing: O=C(Cl)Cl,
[18:45:05] SMILES Parse Error: Failed parsing SMILES 'O=C(Cl)Cl,' for input: 'O=C(Cl)Cl,'
[18:45:05] SMILES Parse Error: syntax error while parsing: O=C(Cl)Cl,
[18:45:05] SMILES Parse Error: Failed parsing SMILES 'O=C(Cl)Cl,' for input: 'O=C(Cl)Cl,'


## Ensure SMILES and mechanism adata are present in the dataframe

In [5]:
def locate_mono_attrs_in_df(monomer_dataframe : pd.DataFrame, monomer_attributes : dict[str, Iterable[str]]) -> Optional[dict[str, str]]:
    '''Takes a dataframe of monomer training data and a dict of desired attributes and the columns in the dataframe it might be found in
    Checks that those columns are present and returns dict with first column for each if all are present, or NoneType otherwise'''
    attr_columns : dict[str, str] = {}

    for targ_attr, col_names_to_check in monomer_attributes.items():
        for col_name in col_names_to_check:
            if col_name in monomer_dataframe:
                attr_columns[targ_attr] = col_name
                break
        else:
            logging.warning(f'Terminated early: no columns found for "{targ_attr}" in column space "{col_names_to_check}"')
            return None # exit early if one of the attribute
        
    return attr_columns

mono_attrs : dict[str, tuple[str]] = { # the attributes to save and the column(s) to check for these values
    'monomer_smiles' : ('smiles_monomer', 'monomer', 'monomers', 'Monomer', 'Monomers'),
    'mechanism'      : ('mechanism', 'rxnname', 'Chemistry')
}

# check each named attribute occurs in at least one columns
attr_locs = locate_mono_attrs_in_df(mono_df, mono_attrs)
assert(attr_locs is not None)
named_attr_cols = [colname for colname in attr_locs.values()]
metadata_cols = ~mono_df.columns.isin(named_attr_cols)

# Parse and reformat SMILES to ensure they are in standard form (i.e. dot-separated string in canonicalized order)
smiles_colname = attr_locs['monomer_smiles']
mono_df[smiles_colname] = mono_df[smiles_colname].map(parse_monomer_smiles)

## Generate jobs for each statepoint, injecting all other parameters not in the dataset

In [119]:
from itertools import product as cartesian_product

def cartesian_grid(param_options : dict[str, Iterable[Any]]) -> Generator[dict[str, Any], None, None]:
    '''
    Takes a dict keyed by parameter names whose values contain
    possible values for each respective parameter
    
    Exhaustively generates dicts (keyed by the same parameter names) containing every
    unique combination of those parameter values, with exactly one value for each key
    '''
    for param_point in cartesian_product(*param_options.values()):
        yield {
            param_name : param_value
                for param_name, param_value in zip(param_options.keys(), param_point)
        }

In [ ]:
gridspec = {
    'DOP' : (3, 5),
    'N_ATOMS_MAX' : (10_000, 20_000)
}
for sp in cartesian_grid(gridspec):
    print(sp)

In [36]:
DOPS : tuple[int, ...] = (3, 5)
N_ATOM_MAX : tuple[int, ...] = (10_000,) # 20_000)

for i, row in mono_df.iterrows():
    statepoint = {}
    
    for attr_name, col_name in attr_locs.items():
        statepoint[attr_name] = row[col_name]
    metadata = row[metadata_cols].replace(np.nan, None).to_dict()

# Skeletal Signac Project

In [ ]:
# Signac workflow control
import signac
from signac import Project
from signac.job import Job

from flow import FlowProject

In [ ]:
class PolyIDBuild(FlowProject):
    pass

In [ ]:
project_path = Path('polyid_test')
project = PolyIDBuild.init_project(project_path)
project.document.rxn_backmap = { # map NREL dataset mechanism names to pre-made rxn template names
    'amide'     : 'polyamide',
    'carbonate' : 'polycarbonate_phosgene',
    'ester'     : 'polyester',
    'imide'     : 'polyimide',
    'urethane'  : 'polyurethane_isocyanate',
    'NIPU'      : 'polyurethane_nonisocyanate',
    'vinyl'     : 'polyvinyl_head_tail'
}

## MD Engine file write

In [ ]:
from abc import ABC, abstractmethod
from openff.interchange import Interchange
from polymerist.genutils.decorators.classmod import register_subclasses


@register_subclasses(key_attr='ENGINE')
class MDEngineExporter(ABC):
    '''For simplifying the process of '''
    def __init_subclass__(cls, **kwargs) -> None:
        '''Enforce class-level definition of "Engine" name attr in subclasses'''
        super().__init_subclass__()
        if not hasattr(cls, 'ENGINE'):
            raise NotImplementedError('No class attr "ENGINE" set for subclass')
        
    def __init__(self, interchange : Interchange) -> None:
        super().__init__()
        self.interchange = interchange

    @property
    def inc(self) -> Interchange:
        '''Alias of "self.interchange" for convenience'''
        return self.interchange
    
    @abstractmethod
    def write_inputs(*args, **kwargs) -> None:
        pass

    
# Concrete classes
class LAMMPSMDExporter(MDEngineExporter):
    ENGINE = 'LAMMPS'

class OpenMMMDExporter(MDEngineExporter):
    ENGINE = 'OpenMM'